# 여러 층을 모아 쓰기: JK-Net, APPNP, GPR-GNN

깊은 층은 평활화되고 얕은 층은 멀리 못 봐요. 그러면 층을 하나만 고르지 말고 다 모아 두었다가 노드마다 좋은 것을 쓰면 되지 않을까요. 그것이 JK-Net 이고, 같은 생각을 전파 쪽에서 밀고 간 것이 APPNP 와 GPR-GNN 이에요. 앞부분은 기출 7번과 같은 모양을 numpy 로 직접 짜 보고, 마지막 실습은 슬라이드 p.65-66 의 JumpingKnowledge 코드를 그대로 써서 Colab 에서 열려요. 5주차는 실습 노트북이 없어서 슬라이드 코드와 기출 계산으로 만들었어요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 준비: 필요한 도구 설치 (처음 한 번만 실행)

In [ ]:
!pip install -q torch_geometric

## 1. PyG 의 JumpingKnowledge 쓰기  (직접 짜기)

슬라이드 p.66 의 JKNet 클래스에서 층 집계기와 마지막 선형층을 만드는 부분을 짜요.

- `from torch_geometric.nn import JumpingKnowledge` 가 슬라이드 맨 윗줄이에요.
- `JumpingKnowledge(mode=...)` 는 `'max'`, `'cat'`, `'lstm'` 세 가지를 받아요. 슬라이드는 앞의 둘을 씁니다.
- `'max'` 는 칸마다 최댓값을 고르므로 크기가 그대로예요. 그래서 마지막 선형층의 입력은 `hidden` 이에요.
- `'cat'` 은 층을 옆으로 이어 붙이므로 입력이 `num_layers * hidden` 이 돼요. 이 한 줄이 두 방식의 비용 차이예요.
- `return jk, fc` 처럼 쉼표로 두 값을 한 번에 돌려줄 수 있어요. 받을 때도 `a, b = ...` 로 풀어 받아요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 방식 이름, 층 수, 은닉 칸 수, 클래스 수예요. 출력은 집계기와 선형층 두 개예요.
2. 방식에 따라 선형층의 입력 칸 수가 달라지는 것이 핵심이에요.
3. `num_layers=6`, `hidden=16` 이면 cat 은 96 칸, max 는 16 칸이에요.
4. 집계기 자체에는 학습할 파라미터가 없어요(lstm 만 예외예요).
5. 실제로 표 여러 장을 넣어 크기가 맞는지 확인해 보면 확실해요.

슈도코드

```text
층 집계기를 방식 이름으로 만든다
방식이 max 이면 입력이 hidden 인 선형층을 만든다
방식이 cat 이면 입력이 num_layers x hidden 인 선형층을 만든다
집계기와 선형층을 함께 돌려준다
```

</details>

<details><summary>힌트 1</summary>

JumpingKnowledge(mode=mode) 로 만들어요

</details>

<details><summary>힌트 2</summary>

if 와 elif 로 두 경우를 나눠요

</details>

<details><summary>힌트 3</summary>

cat 일 때만 num_layers 를 곱해요

</details>

원본: L5 슬라이드 p.65-66 (JumpingKnowledge 예시 코드)


In [ ]:
import torch
from torch_geometric.nn import JumpingKnowledge

# 1. JumpingKnowledge(mode=mode) 로 층 집계기를 만들어요
# 2. mode 가 'max' 면 torch.nn.Linear(hidden, num_classes),
#    'cat' 이면 torch.nn.Linear(num_layers * hidden, num_classes) 를 만들어요
# 3. (jk, fc) 두 개를 돌려주는 함수 make_jk(mode, num_layers, hidden, num_classes) 를 완성해요
def make_jk(mode, num_layers, hidden, num_classes):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
assert "make_jk" in globals(), "make_jk 함수를 만들어야 해요"
r = make_jk('max', 6, 16, 3)
assert r is not None, "return 을 빠뜨렸어요"
jk_m, fc_m = r
assert isinstance(jk_m, JumpingKnowledge), "첫 번째 값은 JumpingKnowledge 여야 해요"
assert fc_m.in_features == 16, f"max 면 선형층 입력이 hidden = 16 이어야 해요. 지금은 {fc_m.in_features} 이에요"
assert fc_m.out_features == 3, "선형층 출력은 클래스 수여야 해요"
jk_c, fc_c = make_jk('cat', 6, 16, 3)
assert fc_c.in_features == 96, f"cat 이면 입력이 6 x 16 = 96 이어야 해요. 지금은 {fc_c.in_features} 이에요"
xs = [torch.randn(4, 16) for _ in range(6)]
assert tuple(jk_m(xs).shape) == (4, 16), "max 집계기의 출력은 4 x 16 이어야 해요"
assert tuple(jk_c(xs).shape) == (4, 96), "cat 집계기의 출력은 4 x 96 이어야 해요"
a = torch.tensor([[1.0, 5.0]])
b = torch.tensor([[3.0, 2.0]])
assert torch.allclose(jk_m([a, b]), torch.tensor([[3.0, 5.0]])), "max 는 칸마다 최댓값을 골라야 해요 (한 층을 통째로 고르는 것이 아니에요)"
print("통과! 슬라이드 p.66 의 JKNet 클래스가 `self.jk = JumpingKnowledge(mode=mode)` 와 `self.fc = ...` 로 정확히 이 일을 해요. 앞에서 numpy 로 짠 jk_max, jk_cat 이 이 한 줄이 된 거예요.")

<details><summary>정답 보기</summary>

```python
import torch
from torch_geometric.nn import JumpingKnowledge


def make_jk(mode, num_layers, hidden, num_classes):
    jk = JumpingKnowledge(mode=mode)
    if mode == 'max':
        fc = torch.nn.Linear(hidden, num_classes)
    elif mode == 'cat':
        fc = torch.nn.Linear(num_layers * hidden, num_classes)
    return jk, fc

```

</details>